# Baseline de prédiction du rendement — `/predict`

**Objectif** : poser une référence simple pour le service `/predict`.

| Modèle | Rôle |
|---|---|
| `DummyRegressor` | référence naïve : prédit toujours le rendement moyen |
| `LinearRegression` | première baseline |


## Imports

In [1]:
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression

from agritech.config import PATHS
from agritech.evaluation import cross_validate_regressor, format_cv_metrics
from agritech.notification import notify
from agritech.preprocessing import PREPROCESSING_DESCRIPTION, build_pipeline, count_encoded_columns
from agritech.tracking import log_run, run_tags, setup_mlflow
from agritech.training_data import (
    PREDICT_BUSINESS_FEATURES,
    PREDICT_CATEGORICAL,
    PREDICT_DATASET,
    PREDICT_FEATURES,
    PREDICT_NUMERIC,
    PREDICT_TARGET,
    load_predict_dataset,
    predict_cv,
    predict_feature_types,
    predict_protocol_params,
    split_predict,
)

# Données

## Lecture et contrôles de base

Dataset produit par le notebook 06. Les 231 rendements négatifs en ont été retirés.

In [2]:
df = load_predict_dataset()

print("fichier            :", PREDICT_DATASET.relative_to(PATHS.root))
print("lignes x colonnes  :", df.shape)
print("valeurs manquantes :", df.isna().sum().sum())
print("rendements < 0     :", (df[PREDICT_TARGET] < 0).sum())
print("\ntypes :")
print(df.dtypes.to_string())

fichier            : data/processed/predict_training_dataset.csv
lignes x colonnes  : (999769, 10)
valeurs manquantes : 0
rendements < 0     : 0

types :
Crop                       object
Soil_Type                  object
Rainfall_mm               float64
Temperature_Celsius       float64
Fertilizer_Used              bool
Irrigation_Used              bool
Region                     object
Weather_Condition          object
Days_to_Harvest             int64
Yield_tons_per_hectare    float64


## Variables candidates

- **Catégorielles** : `Crop`, `Soil_Type`, `Region`, `Weather_Condition`, `Fertilizer_Used`, `Irrigation_Used`.
- **Numériques** : `Rainfall_mm`, `Temperature_Celsius`, `Days_to_Harvest`.

Les 9 variables candidates sont comparées plus bas aux 6 de la configuration métier envisagée, sans `Region`,
`Weather_Condition` ni `Days_to_Harvest`.

In [3]:
print("catégorielles :", PREDICT_CATEGORICAL)
print("numériques    :", PREDICT_NUMERIC)
print("métier        :", PREDICT_BUSINESS_FEATURES)
print("en plus       :", [v for v in PREDICT_FEATURES if v not in PREDICT_BUSINESS_FEATURES])
print("\nmodalités par variable catégorielle :")
print(df[PREDICT_CATEGORICAL].nunique().to_string())

catégorielles : ['Crop', 'Soil_Type', 'Fertilizer_Used', 'Irrigation_Used', 'Region', 'Weather_Condition']
numériques    : ['Rainfall_mm', 'Temperature_Celsius', 'Days_to_Harvest']
métier        : ['Crop', 'Soil_Type', 'Rainfall_mm', 'Temperature_Celsius', 'Fertilizer_Used', 'Irrigation_Used']
en plus       : ['Region', 'Weather_Condition', 'Days_to_Harvest']

modalités par variable catégorielle :
Crop                 6
Soil_Type            6
Fertilizer_Used      2
Irrigation_Used      2
Region               4
Weather_Condition    3


# Découpage train / test

80 % pour l'entraînement, 20 % pour le test, de façon aléatoire et reproductible (`random_state=42`), avant tout
preprocessing appris.

**Le jeu de test est réservé à l'évaluation finale du modèle retenu.**

In [4]:
X_train, X_test, y_train, y_test = split_predict(df)

print(f"total   : {len(df)} lignes")
print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}, réservé à l'évaluation finale")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

assert len(X_train) + len(X_test) == len(df)
assert X_train.index.intersection(X_test.index).empty

total   : 999769 lignes
X_train : (799815, 9)
X_test  : (199954, 9), réservé à l'évaluation finale
y_train : (799815,)
y_test  : (199954,)


# Métriques

- **RMSE** : erreur quadratique moyenne, en t/ha ; elle pénalise davantage les grosses erreurs.
- **R²** : part de la variance du rendement expliquée par le modèle ; 0 pour une prédiction constante, 1 pour une
  prédiction parfaite.
- **MAE** : erreur absolue moyenne, en t/ha.

# Protocole de validation croisée

Les modèles sont comparés par validation croisée à 5 folds, sur le jeu d'entraînement uniquement
(`shuffle=True`, `random_state=42`). Les folds sont les mêmes pour tous les modèles : les scores sont directement
comparables. Ce protocole servira aux modèles `/predict` suivants.

In [5]:
cv = predict_cv()
folds = [(len(apprentissage), len(evaluation)) for apprentissage, evaluation in cv.split(X_train)]

print(cv)
for numero, (n_apprentissage, n_evaluation) in enumerate(folds, start=1):
    print(f"fold {numero} : {n_apprentissage} lignes d'apprentissage | {n_evaluation} d'évaluation")

KFold(n_splits=5, random_state=42, shuffle=True)
fold 1 : 639852 lignes d'apprentissage | 159963 d'évaluation
fold 2 : 639852 lignes d'apprentissage | 159963 d'évaluation
fold 3 : 639852 lignes d'apprentissage | 159963 d'évaluation
fold 4 : 639852 lignes d'apprentissage | 159963 d'évaluation
fold 5 : 639852 lignes d'apprentissage | 159963 d'évaluation


In [6]:
# mêmes tags et mêmes paramètres de protocole pour tous les runs /predict
experience = setup_mlflow("predict")
TAGS = run_tags(service="predict", stage="baseline", notebook="07_predict_training_baseline.ipynb")
PARAMS_PROTOCOLE = predict_protocol_params(X_train, X_test)

expérience MLflow : oc_p12_agritech_predict


# Référence naïve : `DummyRegressor`

`DummyRegressor(strategy="mean")` prédit toujours le rendement moyen. Il fixe le niveau minimal à battre.

In [7]:
dummy = DummyRegressor(strategy="mean")
cv_dummy = cross_validate_regressor(dummy, X_train, y_train, cv)

print(f"écart-type du rendement dans le train : {y_train.std(ddof=0):.3f} t/ha")
print(format_cv_metrics(cv_dummy))

écart-type du rendement dans le train : 1.695 t/ha
RMSE 1.6952 ± 0.0027 t/ha | MAE 1.3884 ± 0.0031 t/ha | R² -0.0000 ± 0.0000


In [8]:
log_run(
    "dummy_baseline_cv",
    TAGS,
    params=PARAMS_PROTOCOLE | {"model": "DummyRegressor", "strategy": "mean", "feature_set": "none", "n_features": 0},
    metrics=cv_dummy,
)

**Observations :**

- La RMSE du `DummyRegressor` (1,695 t/ha) est égale à l'écart-type du rendement.
- Son R² est nul : prédire la moyenne n'explique aucune variation du rendement.

# Régression linéaire — 9 variables candidates

`Pipeline` scikit-learn : catégorielles → `OneHotEncoder`, numériques conservées telles quelles, puis
`LinearRegression`.

C'est le pipeline complet qui est évalué : dans chaque fold, le preprocessing est réappris sur les seules données
d'apprentissage.

In [9]:
lineaire_9 = build_pipeline(LinearRegression(), PREDICT_CATEGORICAL, PREDICT_NUMERIC)
cv_lineaire_9 = cross_validate_regressor(lineaire_9, X_train, y_train, cv)

n_colonnes_9 = count_encoded_columns(PREDICT_CATEGORICAL, PREDICT_NUMERIC, X_train)
print(f"colonnes après encodage : {n_colonnes_9}")
print(format_cv_metrics(cv_lineaire_9))

colonnes après encodage : 26
RMSE 0.5003 ± 0.0009 t/ha | MAE 0.3993 ± 0.0007 t/ha | R² 0.9129 ± 0.0003


In [10]:
log_run(
    "linear_regression_9_features_cv",
    TAGS,
    params=PARAMS_PROTOCOLE | {
        "model": "LinearRegression",
        "feature_set": "candidates_9",
        "n_features": len(PREDICT_FEATURES),
        "n_encoded_features": n_colonnes_9,
        "preprocessing": PREPROCESSING_DESCRIPTION,
    },
    metrics=cv_lineaire_9,
    features={"categorical": PREDICT_CATEGORICAL, "numeric": PREDICT_NUMERIC},
)

**Observations :**

- Les 9 variables deviennent 26 colonnes après encodage.
- RMSE de 0,500 t/ha et R² de 0,913, contre 1,695 t/ha et 0 pour la référence naïve.

# Régression linéaire — 6 variables métier

Même protocole, avec 6 variables : `Region`, `Weather_Condition` et `Days_to_Harvest` sont retirées. L'écart avec le
modèle à 9 variables mesure leur apport.

In [11]:
categorielles_metier, numeriques_metier = predict_feature_types(PREDICT_BUSINESS_FEATURES)

lineaire_6 = build_pipeline(LinearRegression(), categorielles_metier, numeriques_metier)
cv_lineaire_6 = cross_validate_regressor(lineaire_6, X_train[PREDICT_BUSINESS_FEATURES], y_train, cv)

n_colonnes_6 = count_encoded_columns(categorielles_metier, numeriques_metier, X_train)
print(f"colonnes après encodage : {n_colonnes_6}")
print(format_cv_metrics(cv_lineaire_6))

colonnes après encodage : 18
RMSE 0.5003 ± 0.0009 t/ha | MAE 0.3993 ± 0.0007 t/ha | R² 0.9129 ± 0.0003


In [12]:
log_run(
    "linear_regression_6_features_cv",
    TAGS,
    params=PARAMS_PROTOCOLE | {
        "model": "LinearRegression",
        "feature_set": "business_6",
        "n_features": len(PREDICT_BUSINESS_FEATURES),
        "n_encoded_features": n_colonnes_6,
        "preprocessing": PREPROCESSING_DESCRIPTION,
    },
    metrics=cv_lineaire_6,
    features={"categorical": categorielles_metier, "numeric": numeriques_metier},
)

# Comparaison par validation croisée

Les modèles sont comparés sur la RMSE moyenne de validation croisée.

In [13]:
comparaison = pd.DataFrame(
    {
        "DummyRegressor": cv_dummy,
        "LinearRegression — 9 variables": cv_lineaire_9,
        "LinearRegression — 6 variables métier": cv_lineaire_6,
    }
).T.sort_values("cv_rmse_mean")
comparaison.round(4)

,cv_rmse_mean,cv_rmse_std,cv_mae_mean,cv_mae_std,cv_r2_mean,cv_r2_std
LinearRegression — 6 variables métier,0.5003,0.0009,0.3993,0.0007,0.9129,0.0003
LinearRegression — 9 variables,0.5003,0.0009,0.3993,0.0007,0.9129,0.0003
DummyRegressor,1.6952,0.0027,1.3884,0.0031,-0.0000,0.0000


In [14]:
baisse_rmse = 1 - cv_lineaire_9["cv_rmse_mean"] / cv_dummy["cv_rmse_mean"]
ecart_9_6 = cv_lineaire_9["cv_rmse_mean"] - cv_lineaire_6["cv_rmse_mean"]

print(f"RMSE moyenne : {cv_dummy['cv_rmse_mean']:.3f} → {cv_lineaire_9['cv_rmse_mean']:.3f} t/ha, soit {baisse_rmse:.0%} d'erreur en moins")
print(f"écart de RMSE moyenne, 9 − 6 variables : {ecart_9_6:+.6f} t/ha")
print(f"écart-type de la RMSE entre folds       : {cv_lineaire_6['cv_rmse_std']:.6f} t/ha")

RMSE moyenne : 1.695 → 0.500 t/ha, soit 70% d'erreur en moins
écart de RMSE moyenne, 9 − 6 variables : +0.000002 t/ha
écart-type de la RMSE entre folds       : 0.000894 t/ha


**Observations :**

- La régression linéaire réduit la RMSE de 70 % par rapport au `DummyRegressor` et explique 91,3 % de la variance du
  rendement.
- Les scores varient très peu d'un fold à l'autre : l'estimation est stable.
- Les versions à 9 et à 6 variables donnent des scores quasi identiques.

# Conclusion

**Observations :**

- la régression linéaire fournit une bonne baseline ;
- les 3 variables supplémentaires n'apportent pas de gain visible avec ce modèle ;
- la sélection finale sera confirmée avec les modèles suivants ;
- le test reste réservé à l'évaluation finale.

In [15]:
notify(
    "Agritech /predict",
    f"Baseline terminée — RMSE CV {cv_lineaire_9['cv_rmse_mean']:.3f} (9 variables), {cv_lineaire_6['cv_rmse_mean']:.3f} (6 variables)",
)